In [ ]:


This notebook helps to plan a new local business idea before launch.

It is designed for entrepreneurs opening a restaurant, clinic, or gym etc and helps generate a simple launch plan with:
- business overview
- services
- target audience
- strengths
- marketing suggestions
- website and brand direction

This version does not require an existing website or live URL because the business is new.


In [ ]:
import json
import os

import base64
import io
from PIL import Image

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk") and len(api_key) > 10:
    print("API key looks good.")
else:
    print("Please make sure your OPENAI_API_KEY is set correctly in the .env file.")

MODEL = "gpt-4.1-mini"
openai = OpenAI()


API key looks good.


In [ ]:
def generate_business_plan(business_name, business_type, location, target_audience, brand_idea):
    prompt = f"""
Business name: {business_name}
Business type: {business_type}
Location: {location}
Target audience: {target_audience}
Brand idea: {brand_idea}

Create a practical launch plan for this new local business.
Return a polished markdown response with clear headings and bullet points.
Include these sections:
- Business overview
- Services
- Target audience
- Strengths
- Marketing suggestions
- Website and brand direction

Keep it realistic for a local community and launch-ready.
"""

    response = ""
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a smart startup strategist for local businesses."},
            {"role": "user", "content": prompt},
        ],
        stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            yield response

    return response


def launch_gradio_app():
    with gr.Blocks(title="Business Launch Plan Generator") as demo:
        gr.Markdown("# Business Launch Plan Generator")

        with gr.Row():
            chatbot = gr.Chatbot(value=[{"role": "assistant", "content": "Ask me about your business idea and I’ll create a launch plan."}], height=500)
            image_output = gr.Image(height=500, interactive=False)

        with gr.Row():
            business_name = gr.Textbox(label="Business name", placeholder="Enter the business name")
            business_type = gr.Textbox(label="Business type", placeholder="Enter the business type")

        with gr.Row():
            location = gr.Textbox(label="Location", placeholder="Enter the location")
            target_audience = gr.Textbox(label="Target audience", placeholder="Who is the target audience?")

        brand_idea = gr.Textbox(label="Brand idea", placeholder="What is the brand vibe or idea?", lines=3)
        submit = gr.Button("Generate plan & logo")

        def respond(business_name, business_type, location, target_audience, brand_idea):
            # Build textual prompt
            prompt = f"""
Business name: {business_name}
Business type: {business_type}
Location: {location}
Target audience: {target_audience}
Brand idea: {brand_idea}

Create a practical launch plan for this new local business.
Return a polished markdown response with clear headings and bullet points.
Include these sections:
- Business overview
- Services
- Target audience
- Strengths
- Marketing suggestions
- Website and brand direction

Keep it realistic for a local community and launch-ready.
"""

            # Stream text response into the chat
            response = ""
            for chunk in openai.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a smart startup strategist for local businesses."},
                    {"role": "user", "content": prompt},
                ],
                stream=True,
            ):
                delta = chunk.choices[0].delta.content
                if delta:
                    response += delta
                    # yield partial chat message and no image yet
                    yield ([{"role": "assistant", "content": response}], None)

            # After text is complete, generate a logo image via the Images API
            image_prompt = f"Logo for {business_name}, a {business_type}. Clean, minimalist vector logo, single strong color, transparent background, scalable, suitable for signage and social media. Brand idea: {brand_idea}."

            try:
                image_resp = openai.images.generate(model="gpt-image-1", prompt=image_prompt, size="1024x1024")
                b64 = image_resp.data[0].b64_json
                image_bytes = base64.b64decode(b64)
                img = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
            except Exception:
                img = None

            # Final yield with completed text and generated image (or None)
            yield ([{"role": "assistant", "content": response}], img)

        submit.click(
            fn=respond,
            inputs=[business_name, business_type, location, target_audience, brand_idea],
            outputs=[chatbot, image_output],
        )

    return demo


In [13]:
# Launch the Gradio interface

demo = launch_gradio_app()
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://0125be9e27f01c2378.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
